# Crédito pessoal não consignado — avaliação técnica

Análise descritiva do fluxo de concessões e do risco observado no estoque da carteira.
Recorte fechado: março de 2011 a dezembro de 2025. Este notebook usa a base local
versionada e a implementação de `src/`, sem coleta de rede durante a execução.
A atualização opcional é feita por `python -m src.pipeline --atualizar --fim 2025-12-31`.


In [1]:
from pathlib import Path
import sys
import logging
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook_connected'
from IPython.display import display, Markdown
from src.pipeline import executar
from src.analysis import (construir_metricas, analisar_defasagens, testar_robustez,
                          comparar_classificacao_temporal, resumir_qualidade, revisar_sazonalidade)
from src.report import gerar_relatorio
logging.basicConfig(level=logging.INFO)
base, metricas = executar(atualizar=False)
display(resumir_qualidade(base))

INFO:src.pipeline:Usando a base local fechada em dezembro de 2025: C:\Users\migue\Documents\Codex\2026-09-22\quero-atuali\work\analise-inadimplencia-credito-pessoal\data\processed\base_analitica_bcb.csv


INFO:src.pipeline:Gerando métricas e saídas processadas em C:\Users\migue\Documents\Codex\2026-09-22\quero-atuali\work\analise-inadimplencia-credito-pessoal\data\processed.


,inicio,fim,meses,datas_duplicadas,ausencias_total,meses_ausentes,lista_meses_ausentes
0,2011-03-01,2025-12-01,178,0,0,0,Nenhum


## Indicadores e classificação

Concessões e saldo são corrigidos pelo IPCA para preços do último mês. As taxas de
crescimento são calculadas em unidade fixa para não depender do IPCA futuro.
A MM3 usa o mês atual e os dois anteriores; a MM6 usa o atual e cinco anteriores.
O crescimento é `100 × (MM(t)/MM(t−12) − 1)`. A inadimplência usa `D(t) − D(t−12)`, em p.p.
O prêmio bruto é juros menos Selic mensal, não o spread bancário oficial.

A regra principal é retrospectiva: percentil 20 dos movimentos absolutos na amostra inteira.
Valores dentro dos limites inclusivos são estáveis. **Expansão sem deterioração observada**
significa aumento das concessões com risco estável ou em queda na métrica contemporânea;
não garante qualidade das safras novas nem ausência de deterioração futura.


In [2]:
display(metricas[["data", "concessoes_var_12m_pct", "concessoes_reais_var_12m_pct",
                  "inadimplencia_var_12m_pp", "cenario_principal"]].tail(12))
figuras = gerar_relatorio(base, metricas, root / "outputs" / "figures")
display(figuras["crescimento_nominal_real"])
display(figuras["matriz_cenarios_reais_refinada"])
display(figuras["juros_selic"])

,data,concessoes_var_12m_pct,concessoes_reais_var_12m_pct,inadimplencia_var_12m_pp,cenario_principal
166,2025-01-01,30.640298,24.719520,0.46,Expansão sem deterioração observada
167,2025-02-01,28.070671,22.244174,0.88,Crescimento com alerta
168,2025-03-01,21.820644,16.022509,0.88,Crescimento com alerta
169,2025-04-01,16.976382,11.036218,1.19,Crescimento com alerta
170,2025-05-01,14.952787,9.032249,1.38,Crescimento com alerta
171,2025-06-01,16.006522,10.052730,1.77,Crescimento com alerta
172,2025-07-01,16.541888,10.680031,2.07,Crescimento com alerta
173,2025-08-01,14.619863,8.928347,2.63,Crescimento com alerta
174,2025-09-01,14.360794,8.721012,2.29,Crescimento com alerta
175,2025-10-01,15.001046,9.528616,2.51,Crescimento com alerta


## Defasagens de 3, 6, 9 e 12 meses

O alvo principal é a **variação anual avaliada no futuro**: `D(t+h) − D(t+h−12)`.
Para h menor que 12, parte dessa janela antecede t. Por isso, também mostramos a
**mudança exclusivamente entre t e t+h**: `D(t+h) − D(t)`. O preditor descritivo é
o crescimento anual da MM3 real em t. Os últimos h meses não têm alvo observado e
são excluídos, nunca preenchidos. A segunda tabela fixa os mesmos meses de origem
em todos os horizontes. Alvos futuros são usados somente nesta avaliação ex post.
Não há teste causal, p-valor ou seleção do melhor horizonte: autocorrelação,
janelas sobrepostas e choques comuns limitam a leitura de Pearson.


In [3]:
display(analisar_defasagens(metricas))
display(analisar_defasagens(metricas, amostra_comum=True))
display(figuras["defasagens"])

,defasagem_meses,correlacao_pearson,correlacao_mudanca_futura,observacoes,inicio_origem,fim_origem,amostra
0,3,-0.027432,0.211086,161,2012-05-01,2025-09-01,por_horizonte
1,6,0.175698,0.318394,158,2012-05-01,2025-06-01,por_horizonte
2,9,0.339298,0.381477,155,2012-05-01,2025-03-01,por_horizonte
3,12,0.384767,0.384767,152,2012-05-01,2024-12-01,por_horizonte


,defasagem_meses,correlacao_pearson,correlacao_mudanca_futura,observacoes,inicio_origem,fim_origem,amostra
0,3,-0.081000,0.181208,152,2012-05-01,2024-12-01,comum
1,6,0.135294,0.287360,152,2012-05-01,2024-12-01,comum
2,9,0.313835,0.362397,152,2012-05-01,2024-12-01,comum
3,12,0.384767,0.384767,152,2012-05-01,2024-12-01,comum


## Robustez: percentis 10/20/30 e MM3/MM6

Seis combinações, comparadas à referência MM3/P20 no mesmo modo de limite.
O denominador contém somente meses válidos comuns a todas as combinações.
Concordância mede sensibilidade da regra, não acurácia preditiva.
Percentil é um quantil empírico do módulo das variações, não uma faixa fixa de 10/20/30%.


In [4]:
display(testar_robustez(base))
display(testar_robustez(base, modo_limite="expanding"))

,media_movel_meses,percentil_neutro,concordancia_com_referencia_pct,cenarios_distintos,observacoes,modo_limite,divergencias,cenario_ultimo_mes
0,3,10,85.093168,6,161,retrospectivo,24,Crescimento com alerta
1,3,20,100.000000,6,161,retrospectivo,0,Deterioracao do risco
2,3,30,91.304348,6,161,retrospectivo,14,Deterioracao do risco
3,6,10,81.987578,6,161,retrospectivo,29,Crescimento com alerta
4,6,20,90.683230,6,161,retrospectivo,15,Crescimento com alerta
5,6,30,85.714286,6,161,retrospectivo,23,Deterioracao do risco


,media_movel_meses,percentil_neutro,concordancia_com_referencia_pct,cenarios_distintos,observacoes,modo_limite,divergencias,cenario_ultimo_mes
0,3,10,86.131387,6,137,expanding,19,Crescimento com alerta
1,3,20,100.000000,6,137,expanding,0,Deterioracao do risco
2,3,30,95.620438,6,137,expanding,6,Deterioracao do risco
3,6,10,82.481752,6,137,expanding,24,Crescimento com alerta
4,6,20,89.781022,6,137,expanding,14,Crescimento com alerta
5,6,30,88.321168,6,137,expanding,16,Deterioracao do risco


## Look-ahead: limites e informação disponível

Na alternativa expansiva, cada limite usa apenas valores até t−1, após 24
observações válidas. As métricas de t dependem do mês t, portanto só podem ser usadas
depois de publicadas. Os níveis reais em preços finais são retrospectivos, mas
o fator de rebase cancela nas taxas de crescimento. Testes verificam estabilidade
ao truncar a base e ao alterar concessões, inadimplência e IPCA futuros.

Esta base não contém vintages nem datas de divulgação. Logo, a janela expansiva
remove informação futura da estimação de limites, mas não é um backtest operacional
em tempo real e não elimina vazamento por revisões das séries.


In [5]:
comparacao, resumo = comparar_classificacao_temporal(base)
display(resumo)
display(comparacao.tail(12))
prefixo = construir_metricas(base.iloc[:100], modo_limite="expanding")
completa = construir_metricas(base, modo_limite="expanding")
pd.testing.assert_series_equal(prefixo.cenario_principal, completa.cenario_principal.iloc[:100])

,observacoes_comparaveis,concordancia_pct,divergencias
0,140,95.714286,6


,data,cenario_retrospectivo,cenario_sem_lookahead,classificacoes_concordam
166,2025-01-01,Expansão sem deterioração observada,Expansão sem deterioração observada,True
167,2025-02-01,Crescimento com alerta,Crescimento com alerta,True
168,2025-03-01,Crescimento com alerta,Crescimento com alerta,True
169,2025-04-01,Crescimento com alerta,Crescimento com alerta,True
170,2025-05-01,Crescimento com alerta,Crescimento com alerta,True
171,2025-06-01,Crescimento com alerta,Crescimento com alerta,True
172,2025-07-01,Crescimento com alerta,Crescimento com alerta,True
173,2025-08-01,Crescimento com alerta,Crescimento com alerta,True
174,2025-09-01,Crescimento com alerta,Crescimento com alerta,True
175,2025-10-01,Crescimento com alerta,Crescimento com alerta,True


## Revisão de sazonalidade

Comparamos o perfil por mês do calendário da variação mensal real e das variações
anuais das MM3/MM6, em amostra comum. As médias móveis são retrospectivas, nunca
centradas. Comparar o mesmo mês do ano anterior atenua padrões sazonais estáveis,
mas não constitui ajuste sazonal formal. Efeitos de calendário, tendência,
pandemia e mudanças de composição podem permanecer. O perfil abaixo é diagnóstico,
não um teste estatístico de sazonalidade. Não foram aplicados ADF/KPSS, e não se
afirma estacionariedade. A suavização e a diferenciação anual induzem sobreposição.


In [6]:
display(revisar_sazonalidade(metricas))
display(figuras["sazonalidade"])

,mes_calendario,observacoes,mensal_media_pct,anual_mm3_media_pct,anual_mm6_media_pct
0,1,13,3.472225,4.182702,4.171122
1,2,13,-10.047556,4.433366,4.231388
2,3,13,12.989551,3.990104,3.983073
3,4,13,-2.626122,3.315999,3.647588
4,5,13,3.567818,2.719143,3.425439
5,6,13,-0.816017,2.876760,3.307496
6,7,13,-0.834377,3.388488,3.182913
7,8,14,3.620158,3.737560,3.578308
8,9,14,-2.213429,4.394244,3.849911
9,10,14,4.077908,4.624176,4.121440


## Conclusões e limitações

A síntese abaixo é gerada a partir dos resultados executados. O consignado é
referência descritiva, não contrafactual. Dados agregados não permitem avaliar
risco individual, coortes de originação ou atribuir causalidade entre fluxo e estoque.


In [7]:
display(Markdown((root / "outputs" / "conclusoes.md").read_text(encoding="utf-8")))

# Conclusões — 12/2025

Concessões: R$ 21.75 bilhões. Crescimento anual da MM3: 8.22% nominal e 3.59% real. Inadimplência: 9.16%, variação anual de 2.70 p.p. Cenário retrospectivo: **Deterioracao do risco**.

Robustez retrospectiva: concordância de 82.0% a 100.0% em 161 meses comuns. Retrospectiva × expansiva: 95.7% em 140 meses.

## Defasagens em amostra comum

Horizonte | Correlação com variação anual futura | Correlação com mudança entre t e t+h | Pares
---: | ---: | ---: | ---:
3 | -0.081 | 0.181 | 152
6 | 0.135 | 0.287 | 152
9 | 0.314 | 0.362 | 152
12 | 0.385 | 0.385 | 152

As concessões permaneceram ativas, mas houve piora do risco observado no estoque. A correlação varia com o horizonte e a definição do alvo; não permite atribuir essa piora às novas operações. Janelas anuais se sobrepõem, e autocorrelação e choques comuns impedem interpretar os pares como observações independentes.

O perfil mensal é diagnóstico descritivo, sem ajuste sazonal formal ou comprovação de estacionariedade. Expansão sem deterioração observada descreve apenas o risco agregado contemporâneo, sem garantir qualidade futura. A versão expansiva exclui o mês atual da estimação dos limites, mas não simula vintages e atrasos de publicação.
